In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

# ---------------- Settings ----------------

ROOT = Path(
    r"E:\NO-NC\normalization\revise\reviewer3_threshold_results"
)
OUT_CSV = ROOT / "paired_oof_model_comparisons.csv"

THRESHOLDS = [5, 10, 50]
MODELS = [
    "RandomForestRegressor",
    "XGBoostRegressor",
    "CatBoostRegressor",
    "LightGBMRegressor",
    "SVR",
    "MLPRegressor",
    "LogisticRegression",
]

N_BOOTSTRAP = 5000
ALPHA = 0.05
RANDOM_STATE = 20261101


def paired_stratified_bootstrap(y, scores_a, scores_b, n_bootstrap, rng):
    """
    对相同观测的两组连续预测分数做配对、分层 bootstrap。
    每次在阳性和阴性样本内分别抽样，保持原有类别数量。
    返回 A-B 的 ROC-AUC 与 Average Precision 差值分布。
    """
    y = np.asarray(y, dtype=int)
    scores_a = np.asarray(scores_a, dtype=float)
    scores_b = np.asarray(scores_b, dtype=float)

    pos = np.flatnonzero(y == 1)
    neg = np.flatnonzero(y == 0)
    if len(pos) == 0 or len(neg) == 0:
        raise ValueError("OOF 数据必须同时包含阳性和阴性样本。")

    auc_diff = np.empty(n_bootstrap)
    ap_diff = np.empty(n_bootstrap)

    for b in range(n_bootstrap):
        idx = np.concatenate([
            rng.choice(pos, size=len(pos), replace=True),
            rng.choice(neg, size=len(neg), replace=True),
        ])
        rng.shuffle(idx)

        y_b = y[idx]
        auc_diff[b] = (
            roc_auc_score(y_b, scores_a[idx])
            - roc_auc_score(y_b, scores_b[idx])
        )
        ap_diff[b] = (
            average_precision_score(y_b, scores_a[idx])
            - average_precision_score(y_b, scores_b[idx])
        )

    return auc_diff, ap_diff


records = []

for threshold in THRESHOLDS:
    folder = ROOT / f"threshold_{threshold}"
    best = pd.read_csv(folder / "best_model_by_threshold.csv").iloc[0]
    winner = str(best["model"])
    feature_set = str(best["feature_set"])

    # 读取该阈值、该特征方案下各模型的 OOF 预测
    pred = {}
    for model in MODELS:
        path = (
            folder / "per_model_results"
            / f"{model}_oof_predictions.csv"
        )
        if not path.exists():
            raise FileNotFoundError(path)

        df = pd.read_csv(path)
        df = df.loc[df["feature_set"].astype(str) == feature_set].copy()
        df = df.sort_values("train_sample_index").reset_index(drop=True)

        required = {
            "train_sample_index", "y_true", "oof_score", "outer_fold"
        }
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"{path} 缺少列：{sorted(missing)}")
        if df["train_sample_index"].duplicated().any():
            raise ValueError(f"{path} 中 train_sample_index 不唯一。")

        pred[model] = df

    # 确认所有模型预测对应相同观测、标签和外层折
    ref = pred[winner]
    for model, df in pred.items():
        if not np.array_equal(
            ref["train_sample_index"].to_numpy(),
            df["train_sample_index"].to_numpy()
        ):
            raise ValueError(
                f"{threshold} μg/L：{model} 与 {winner} 的样本索引未对齐。"
            )
        if not np.array_equal(
            ref["y_true"].to_numpy(),
            df["y_true"].to_numpy()
        ):
            raise ValueError(
                f"{threshold} μg/L：{model} 与 {winner} 的标签未对齐。"
            )
        if not np.array_equal(
            ref["outer_fold"].to_numpy(),
            df["outer_fold"].to_numpy()
        ):
            raise ValueError(
                f"{threshold} μg/L：{model} 与 {winner} 的外层折未对齐。"
            )

    y = ref["y_true"].to_numpy(dtype=int)
    winner_score = ref["oof_score"].to_numpy(dtype=float)
    competitors = [m for m in MODELS if m != winner]

    # 每个阈值有 6 组比较；Bonferroni 校正后的双侧区间
    n_comparisons = len(competitors)
    tail_probability = ALPHA / (2 * n_comparisons)

    print(
        f"{threshold} μg/L: selected={winner}, "
        f"feature_set={feature_set}, n={len(y)}, "
        f"positive={int(y.sum())}"
    )

    for i, model in enumerate(competitors):
        other_score = pred[model]["oof_score"].to_numpy(dtype=float)

        point_roc_diff = (
            roc_auc_score(y, winner_score)
            - roc_auc_score(y, other_score)
        )
        point_pr_diff = (
            average_precision_score(y, winner_score)
            - average_precision_score(y, other_score)
        )

        rng = np.random.default_rng(
            RANDOM_STATE + threshold * 100 + i
        )
        roc_diffs, pr_diffs = paired_stratified_bootstrap(
            y=y,
            scores_a=winner_score,
            scores_b=other_score,
            n_bootstrap=N_BOOTSTRAP,
            rng=rng,
        )

        roc_low, roc_high = np.quantile(
            roc_diffs, [tail_probability, 1 - tail_probability]
        )
        pr_low, pr_high = np.quantile(
            pr_diffs, [tail_probability, 1 - tail_probability]
        )

        records.append({
            "target_threshold_ug_L": threshold,
            "feature_set": feature_set,
            "reference_model": winner,
            "comparison_model": model,
            "n_oof": len(y),
            "n_positive_oof": int(y.sum()),
            "delta_ROC_AUC_reference_minus_comparison": point_roc_diff,
            "ROC_AUC_diff_CI_lower_Bonferroni": roc_low,
            "ROC_AUC_diff_CI_upper_Bonferroni": roc_high,
            "ROC_AUC_difference_supported": bool(
                roc_low > 0 or roc_high < 0
            ),
            "delta_PR_AUC_reference_minus_comparison": point_pr_diff,
            "PR_AUC_diff_CI_lower_Bonferroni": pr_low,
            "PR_AUC_diff_CI_upper_Bonferroni": pr_high,
            "PR_AUC_difference_supported": bool(
                pr_low > 0 or pr_high < 0
            ),
            "bootstrap_replicates": N_BOOTSTRAP,
            "CI_method": (
                "paired stratified percentile bootstrap; "
                "Bonferroni-adjusted across six comparisons"
            ),
        })

results = pd.DataFrame(records)
results.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

print(f"\n已保存配对比较结果：{OUT_CSV}")
print(results.to_string(index=False))

C:\ProgramData\Anaconda3\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
C:\ProgramData\Anaconda3\lib\site-packages\numpy\.libs\libopenblas.JPIJNSWNNAN3CE6LLI5FWSPHUT2VXMTH.gfortran-win_amd64.dll
C:\ProgramData\Anaconda3\lib\site-packages\numpy\.libs\libopenblas.WCDJNK7YVMPZQ2ME2ZZHJJRJ3JIKNDB7.gfortran-win_amd64.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


5 μg/L: selected=RandomForestRegressor, feature_set=all, n=3230, positive=470
10 μg/L: selected=XGBoostRegressor, feature_set=SelectKBest, n=3230, positive=206
50 μg/L: selected=LightGBMRegressor, feature_set=correlation, n=3230, positive=38

已保存配对比较结果：E:\NO-NC\normalization\revise\reviewer3_threshold_results\paired_oof_model_comparisons.csv
 target_threshold_ug_L feature_set       reference_model      comparison_model  n_oof  n_positive_oof  delta_ROC_AUC_reference_minus_comparison  ROC_AUC_diff_CI_lower_Bonferroni  ROC_AUC_diff_CI_upper_Bonferroni  ROC_AUC_difference_supported  delta_PR_AUC_reference_minus_comparison  PR_AUC_diff_CI_lower_Bonferroni  PR_AUC_diff_CI_upper_Bonferroni  PR_AUC_difference_supported  bootstrap_replicates                                                                          CI_method
                     5         all RandomForestRegressor      XGBoostRegressor   3230             470                                  0.003901                         -0.00

In [2]:
from pathlib import Path

import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import (
    Alignment,
    Border,
    Font,
    PatternFill,
    Side,
)
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.worksheet.page import PageMargins


# -------------------- 输入与输出 --------------------

ROOT = Path(
    r"E:\NO-NC\normalization\revise\reviewer3_threshold_results"
)
INPUT_CSV = ROOT / "paired_oof_model_comparisons.csv"
OUTPUT_XLSX = ROOT / "paired_oof_model_comparisons_TableS5.xlsx"


# -------------------- 读取并整理数据 --------------------

df = pd.read_csv(INPUT_CSV)

required_columns = [
    "target_threshold_ug_L",
    "feature_set",
    "reference_model",
    "comparison_model",
    "n_oof",
    "n_positive_oof",
    "delta_ROC_AUC_reference_minus_comparison",
    "ROC_AUC_diff_CI_lower_Bonferroni",
    "ROC_AUC_diff_CI_upper_Bonferroni",
    "delta_PR_AUC_reference_minus_comparison",
    "PR_AUC_diff_CI_lower_Bonferroni",
    "PR_AUC_diff_CI_upper_Bonferroni",
]
missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise ValueError(f"输入 CSV 缺少这些列：{missing}")

if len(df) != 18:
    raise ValueError(
        f"预期 18 组模型比较，实际读取到 {len(df)} 行；请检查输入文件。"
    )

model_labels = {
    "RandomForestRegressor": "Random Forest",
    "XGBoostRegressor": "XGBoost",
    "CatBoostRegressor": "CatBoost",
    "LightGBMRegressor": "LightGBM",
    "SVR": "SVR",
    "MLPRegressor": "MLP",
    "LogisticRegression": "Logistic Regression",
}
feature_labels = {
    "all": "All predictors",
    "SelectKBest": "SelectKBest",
    "correlation": "Correlation filter",
    "PFI": "PFI",
}

def format_diff(estimate, lower, upper):
    """格式：0.004 (−0.009, 0.018)"""
    return (
        f"{estimate:.3f} "
        f"({lower:.3f}, {upper:.3f})"
    )

rows = []
for _, r in df.iterrows():
    roc_text = format_diff(
        r["delta_ROC_AUC_reference_minus_comparison"],
        r["ROC_AUC_diff_CI_lower_Bonferroni"],
        r["ROC_AUC_diff_CI_upper_Bonferroni"],
    )
    pr_text = format_diff(
        r["delta_PR_AUC_reference_minus_comparison"],
        r["PR_AUC_diff_CI_lower_Bonferroni"],
        r["PR_AUC_diff_CI_upper_Bonferroni"],
    )

    rows.append([
        f'{int(r["target_threshold_ug_L"])} μg/L',
        model_labels.get(r["reference_model"], r["reference_model"]),
        model_labels.get(r["comparison_model"], r["comparison_model"]),
        feature_labels.get(r["feature_set"], r["feature_set"]),
        f'{int(r["n_oof"])} ({int(r["n_positive_oof"])})',
        roc_text,
        pr_text,
    ])


# -------------------- 创建论文格式工作簿 --------------------

wb = Workbook()
ws = wb.active
ws.title = "Table S5"

headers = [
    "As threshold",
    "Selected model",
    "Comparator",
    "Feature scheme",
    "OOF n (positive)",
    "ΔROC-AUC (95% CI)",
    "ΔPR-AUC (95% CI)",
]

# 标题
ws.merge_cells("A1:G1")
ws["A1"] = (
    "Table S5. Paired bootstrap comparisons of model performance "
    "using out-of-fold predictions"
)
ws["A1"].font = Font(name="Times New Roman", size=12, bold=True)
ws["A1"].alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)
ws.row_dimensions[1].height = 34

# 说明行
ws.merge_cells("A2:G2")
ws["A2"] = (
    "Differences are calculated as selected model minus comparator; "
    "positive values favor the selected model."
)
ws["A2"].font = Font(name="Times New Roman", size=10, italic=True)
ws["A2"].alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)
ws.row_dimensions[2].height = 30

# 表头
header_row = 4
for col, value in enumerate(headers, start=1):
    cell = ws.cell(row=header_row, column=col, value=value)
    cell.font = Font(name="Times New Roman", size=10, bold=True, color="FFFFFF")
    cell.fill = PatternFill("solid", fgColor="365F91")
    cell.alignment = Alignment(
        horizontal="center",
        vertical="center",
        wrap_text=True,
    )

# 数据行
first_data_row = header_row + 1
for row_idx, row_values in enumerate(rows, start=first_data_row):
    for col_idx, value in enumerate(row_values, start=1):
        cell = ws.cell(row=row_idx, column=col_idx, value=value)
        cell.font = Font(name="Times New Roman", size=10)
        cell.alignment = Alignment(
            horizontal="center" if col_idx not in (2, 3, 4) else "left",
            vertical="center",
            wrap_text=True,
        )
        if (row_idx - first_data_row) % 2 == 1:
            cell.fill = PatternFill("solid", fgColor="EAF0F6")

# 单元格边框
thin_gray = Side(style="thin", color="B7C9D6")
for row in ws.iter_rows(
    min_row=header_row,
    max_row=first_data_row + len(rows) - 1,
    min_col=1,
    max_col=len(headers),
):
    for cell in row:
        cell.border = Border(
            left=thin_gray,
            right=thin_gray,
            top=thin_gray,
            bottom=thin_gray,
        )

# 表格对象：筛选与交替行样式
last_data_row = first_data_row + len(rows) - 1
tab = Table(
    displayName="PairedOOFComparisons",
    ref=f"A{header_row}:G{last_data_row}",
)
tab.tableStyleInfo = TableStyleInfo(
    name="TableStyleMedium2",
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=False,
)
ws.add_table(tab)

# 表注
footnote_row = last_data_row + 2
ws.merge_cells(start_row=footnote_row, start_column=1,
               end_row=footnote_row + 2, end_column=7)
ws.cell(
    row=footnote_row,
    column=1,
    value=(
        "Note: OOF denotes out-of-fold. Confidence intervals are "
        "Bonferroni-adjusted for six model comparisons within each "
        "threshold and metric. They were obtained from 5,000 paired, "
        "class-stratified percentile bootstrap resamples of the matched "
        "OOF predictions. An interval excluding zero indicates a "
        "difference supported after this adjustment. These comparisons "
        "are based on training-set OOF predictions, not independent-test "
        "predictions."
    ),
)
note = ws.cell(row=footnote_row, column=1)
note.font = Font(name="Times New Roman", size=9, italic=True)
note.alignment = Alignment(
    horizontal="left",
    vertical="top",
    wrap_text=True,
)

# 页面与列宽
widths = {
    "A": 15,
    "B": 22,
    "C": 22,
    "D": 20,
    "E": 16,
    "F": 27,
    "G": 27,
}
for col, width in widths.items():
    ws.column_dimensions[col].width = width

ws.row_dimensions[header_row].height = 36
for r in range(first_data_row, last_data_row + 1):
    ws.row_dimensions[r].height = 28

ws.freeze_panes = "A5"
ws.sheet_view.showGridLines = False
ws.auto_filter.ref = f"A{header_row}:G{last_data_row}"

ws.page_setup.orientation = "landscape"
ws.page_setup.fitToWidth = 1
ws.page_setup.fitToHeight = 0
ws.sheet_properties.pageSetUpPr.fitToPage = True
ws.page_margins = PageMargins(
    left=0.25, right=0.25, top=0.5, bottom=0.5,
    header=0.2, footer=0.2,
)
ws.print_title_rows = f"1:{header_row}"
ws.print_area = f"A1:G{footnote_row + 2}"

wb.save(OUTPUT_XLSX)
print(f"Saved: {OUTPUT_XLSX}")

Saved: E:\NO-NC\normalization\revise\reviewer3_threshold_results\paired_oof_model_comparisons_TableS5.xlsx
